## Technical Lesson: Advanced Prompt Engineering with LLMs


In [143]:
### Step 1: Set Up the Environment and Import Libraries

In [144]:
import requests
import os
import json

# Get API token — works in Google Colab (via Secrets) or local environments
try:
    from google.colab import userdata
    api_token = userdata.get('HF_TOKEN')
    print('Loaded HF_TOKEN from Colab Secrets.')
except (ImportError, ModuleNotFoundError):
    # Fallback for local/non-Colab environments
    api_token = os.getenv('HF_TOKEN')
    if api_token:
        print('Loaded HF_TOKEN from environment variable.')
    else:
        print('WARNING: No HF_TOKEN found. Set it as an env var or add it to Colab Secrets.')


Loaded HF_TOKEN from environment variable.


### Step 2: Define the API Endpoint and Request Headers

**Note:** Hugging Face has transitioned from the legacy Serverless Inference API
(`api-inference.huggingface.co`) to the new **Inference Providers** system.
The new system uses an OpenAI-compatible chat completions endpoint at
`router.huggingface.co` and routes requests through third-party providers
(e.g., Novita, SambaNova, Together AI). Every HF account receives free
monthly credits to experiment with.


In [145]:
# Hugging Face Inference Providers — OpenAI-compatible endpoint
API_URL = "https://router.huggingface.co/v1/chat/completions"

# We use Qwen2.5-7B-Instruct, a strong open-source instruct model
# comparable in size to the Mistral-7B model used previously.
# Browse available models at: https://huggingface.co/models?inference=warm&pipeline_tag=text-generation
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# Set up headers for the API request
headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
}

In [146]:
# Define an advanced prompt with in-context examples and chain-of-thought cues
system_prompt = "You are an expert technical assistant."
user_prompt = """
When asked a question about debugging Python code, first, analyze the problem step-by-step and then provide a clear solution.

Example:
Q: I'm getting a 'TypeError: unsupported operand type(s)' when trying to add a string and an integer.
A: First, check where the addition is performed. The error indicates that one operand is a string and the other is an integer.
   Make sure both operands are of the same type, or convert one using int() or str() as needed.

Now, answer the following:
Q: I have a function that returns None, but I expected a number. Why might this be happening?
"""
#A: First, this usually happens when the function does not explicitly return a value in all execution paths. In Python, if a function finishes without hitting a return statement, it automatically returns None. 
# Another possible reason is that a return statement exists but is inside a conditional block that is not being executed, so the function falls through and returns None by default.

#To fix this, ensure that:
     # Every logical path in the function includes a return statement
     # Conditional branches are correctly structured
     # Debugging prints or checks confirm that the expected branch is executed

# In summary, the function is likely missing a proper return statement in one or more execution paths, causing it to return None instead of a numeric value."""

### Step 4: Define the Query Function to Call the Hugging Face API

The new Inference Providers API uses the **OpenAI chat completions format**,
which structures the conversation as a list of `{"role": ..., "content": ...}`
messages rather than the raw `[INST]` template used by the legacy API.


In [147]:
def query_huggingface_api():
    try:
        # OpenAI-compatible chat completions payload
        payload = {
            "model": MODEL_ID,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "max_tokens": 250,
            "temperature": 0.3
        }

        # Make the API request
        response = requests.post(API_URL, headers=headers, json=payload)

        # Raise error if request failed
        response.raise_for_status()

        # Parse response safely
        result = response.json()
        return result["choices"][0]["message"]["content"]

    except requests.exceptions.HTTPError as http_err:
        status_code = http_err.response.status_code if http_err.response else None

        if status_code == 401:
            return "Authentication error: Please check your Hugging Face API token."
        elif status_code == 402:
            return "Payment required: You may have exhausted your free credits."
        elif status_code == 503:
            return "Model is loading. Please retry shortly."
        else:
            return f"HTTP error occurred: {http_err}"

    except Exception as e:
        print(f"Error calling Hugging Face API: {e}")

        # fallback response (safe default)
        return (
            "A function might return None instead of a number for several reasons:\n\n"
            "1. Missing return statement\n"
            "2. Conditional logic skipping return\n"
            "3. Exceptions being caught\n"
            "4. Input validation failures\n"
            "5. Uninitialized variables"
        )

In [148]:
### Step 5: Execute the Query and Display the Output

def run_query():
    result = None  # initialize variable

    try:
        # Combine system + user prompt (if using HF or LLM API)
        full_prompt = system_prompt + "\n\n" + user_prompt

        # Call your API function (must return model output)
        result = query_huggingface_api(full_prompt)

        # Safety check for empty response
        if not result:
            result = "Query failed: Empty response returned"

    except Exception as e:
        result = f"Query failed: {str(e)}"

    return result


# Step 5: Execute the Query and Display the Output
output = run_query()
print("Output:", output)

Output: Query failed: query_huggingface_api() takes 0 positional arguments but 1 was given


In [149]:
# Get response (either from API or fallback)
generated_output = query_huggingface_api()
print("Generated Response:", generated_output)


Generated Response: HTTP error occurred: 401 Client Error: Unauthorized for url: https://router.huggingface.co/v1/chat/completions
